In [34]:


# ================== CELL 1: Global Settings and Environment Setup ==================
import os
import time
import random
import gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from scipy.stats import gaussian_kde

# --- Core Configuration ---
class Config:
    USE_GOOGLE_DRIVE = True
    DRIVE_PATH = '/content/drive/MyDrive/Colab/VisionTransformer_new_2'
    LEARNING_RATE = 9e-4
    SEED = 16

cfg = Config()

# --- Environment Setup ---
if cfg.USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = str(Path(cfg.DRIVE_PATH) / 'data')
    OUTPUT_ROOT = str(Path(cfg.DRIVE_PATH) / 'outputs')
else:
    DATA_ROOT = './data'
    OUTPUT_ROOT = './outputs'

Path(DATA_ROOT).mkdir(parents=True, exist_ok=True);
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"[INFO] Global seed set to {seed}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {device}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[INFO] Using device: cuda


In [35]:

# ================== CELL 2: Model Architecture Definition ==================
class AttnSaveEncoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=256, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.activation = nn.ReLU()
        self.attn_map = None
        self.hidden_state = None
    def forward(self, src):
        attn_output, attn_weights = self.self_attn(src, src, src, need_weights=True, average_attn_weights=False)
        self.attn_map = attn_weights.detach().cpu()
        src = src + self.dropout1(attn_output)
        src = self.norm1(src)
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src))))
        src = src + self.dropout2(src2)
        src = self.norm2(src)
        self.hidden_state = src.detach().cpu()
        return src
class PatchEmbed(nn.Module):
    def __init__(self, img_size, patch_size, in_chans, embed_dim):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
    def forward(self, x):
        x = self.proj(x); x = x.flatten(2); x = x.transpose(1, 2); return x
class SimpleTransformerClassifier(nn.Module):
    def __init__(self, img_size, patch_size, in_chans, n_heads, n_layers, hidden_dim, n_classes):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, in_chans, hidden_dim)
        self.n_patches = self.patch_embed.n_patches
        self.pos_embed = nn.Parameter(torch.zeros(1, self.n_patches, hidden_dim))
        self.encoder_layers = nn.ModuleList([AttnSaveEncoderLayer(hidden_dim, n_heads) for _ in range(n_layers)])
        self.classifier = nn.Linear(hidden_dim * self.n_patches, n_classes)
        self.latest_attn_maps = None
        self.latest_hidden_states = None
    def forward(self, x):
        x = self.patch_embed(x) + self.pos_embed
        attn_maps, hidden_states = [], []
        for layer in self.encoder_layers:
            x = layer(x)
            attn_maps.append(layer.attn_map)
            hidden_states.append(layer.hidden_state)
        out = x.contiguous().view(x.size(0), -1)
        self.latest_attn_maps = attn_maps
        self.latest_hidden_states = hidden_states
        return self.classifier(out)


In [36]:

# ================== CELL 3: Core Functions ==================
def robust_grad_entropy(grad_tensor: torch.Tensor, num_bins=64) -> float:
    if not isinstance(grad_tensor, np.ndarray): g = np.abs(grad_tensor.numpy()).ravel()
    else: g = np.abs(grad_tensor).ravel()
    g = np.log(g + 1e-12); hist, _ = np.histogram(g, bins=num_bins)
    p = hist / max(hist.sum(), 1e-12); p = p[p > 0]
    if len(p) <= 1: return 0.0
    H = -np.sum(p * np.log2(p + 1e-12)); H_norm = H / np.log2(len(p)); return H_norm
def robust_attention_entropy(attn_map: np.ndarray, kde_bw=0.05) -> np.ndarray:
    if attn_map.ndim == 4: attn_map = attn_map.mean(axis=1)
    entropies = []
    for single_map in attn_map:
        flat_dist = single_map.flatten()
        if len(flat_dist) < 2 or np.std(flat_dist) < 1e-9: entropies.append(0.0); continue
        try:
            kde = gaussian_kde(flat_dist, bw_method=kde_bw); x_grid = np.linspace(flat_dist.min(), flat_dist.max(), 100)
            p = kde.evaluate(x_grid); p /= p.sum(); eps = 1e-12
            H = -np.sum(p * np.log2(p + eps)); entropies.append(H)
        except np.linalg.LinAlgError: entropies.append(0.0)
    return np.mean(entropies)
def topk_mass(attn_map: np.ndarray, k=1) -> float:
    if attn_map.ndim == 4: attn_map = attn_map.mean(axis=1)
    sorted_vals = np.sort(attn_map, axis=-1); topk_sum = sorted_vals[..., -k:].sum(axis=-1); return float(topk_sum.mean())
def hidden_activation_entropy(hidden_states_np: np.ndarray) -> float:
    X = hidden_states_np.reshape(-1, hidden_states_np.shape[-1]); X = (X - X.mean(0)) / (X.std(0) + 1e-6)
    bins = np.linspace(-3, 3, 64); ent_list = []
    for d in range(X.shape[1]):
        hist, _ = np.histogram(X[:, d], bins=bins, density=True); p = hist / (hist.sum() + 1e-12); p = p[p > 0]
        if len(p) == 0: continue
        ent = -np.sum(p * np.log2(p + 1e-12)); ent_list.append(ent)
    return np.mean(ent_list) if ent_list else 0.0
def logits_margin(logits: np.ndarray, labels: np.ndarray) -> float:
    idx = np.arange(len(labels)); correct_logits = logits[idx, labels]; masked = logits.copy()
    masked[idx, labels] = -1e9; max_other_logits = masked.max(axis=1); return float((correct_logits - max_other_logits).mean())
def head_similarity_matrix(attn_layer_np: np.ndarray):
    X = attn_layer_np.reshape(attn_layer_np.shape[0], -1); norm = np.linalg.norm(X, axis=1, keepdims=True)
    X_normalized = X / (norm + 1e-12); return X_normalized @ X_normalized.T
def specialization_score(attn_map_bhnn: np.ndarray):
    batch_scores = []
    for i in range(attn_map_bhnn.shape[0]):
        sim_mat = head_similarity_matrix(attn_map_bhnn[i]); H = sim_mat.shape[0]
        if H <= 1: continue
        mean_sim = (sim_mat.sum() - np.trace(sim_mat)) / (H * H - H); batch_scores.append(1.0 - float(mean_sim))
    return np.mean(batch_scores) if batch_scores else 0.0
def get_flat_grad(model):
    grads = []
    for p in model.parameters():
        if p.grad is not None and p.requires_grad: grads.append(p.grad.detach().flatten().cpu())
    return torch.cat(grads) if grads else torch.tensor([], dtype=torch.float32)
def recent_slope(data_series, window=5):
    if len(data_series) < window: return -1.0
    y = np.array(data_series[-window:]); x = np.arange(len(y)); slope = np.polyfit(x, y, deg=1)[0]; return float(slope)
@torch.no_grad()
def evaluate_simple(model, data_loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, total_correct, total = 0.0, 0, 0
    for images, labels in data_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images); loss = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0); preds = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item(); total += labels.size(0)
    return (total_loss / total if total > 0 else 0), (total_correct / total if total > 0 else 0)
@torch.no_grad()
def evaluate_full_metrics(model, data_loader, device):
    model.eval(); num_layers = len(model.encoder_layers)
    batch_metrics = {
        'attn_entropy': [[] for _ in range(num_layers)], 'top1_mass': [[] for _ in range(num_layers)],
        'hidden_entropy': [[] for _ in range(num_layers)], 'head_specialization': [[] for _ in range(num_layers)],
    }
    all_logits, all_labels = [], []
    for images, labels in tqdm(data_loader, desc="Full Evaluation", leave=False):
        images, labels_tensor = images.to(device), labels.to(device)
        outputs = model(images)
        all_logits.append(outputs.cpu().numpy()); all_labels.append(labels.cpu().numpy())
        if model.latest_attn_maps:
            for l in range(num_layers):
                batch_metrics['attn_entropy'][l].append(robust_attention_entropy(model.latest_attn_maps[l].numpy()))
                batch_metrics['top1_mass'][l].append(topk_mass(model.latest_attn_maps[l].numpy(), k=1))
                batch_metrics['head_specialization'][l].append(specialization_score(model.latest_attn_maps[l].numpy()))
        if model.latest_hidden_states:
             for l in range(num_layers):
                batch_metrics['hidden_entropy'][l].append(hidden_activation_entropy(model.latest_hidden_states[l].numpy()))
    epoch_metrics = {}
    for key, layer_values in batch_metrics.items():
        epoch_metrics[key] = [np.median(vals) if vals else 0.0 for vals in layer_values]
    final_logits = np.concatenate(all_logits, axis=0); final_labels = np.concatenate(all_labels, axis=0)
    epoch_metrics['logits_margin'] = logits_margin(final_logits, final_labels)
    return epoch_metrics
@torch.no_grad()
def get_coarse_global_attn_entropy(model, data_loader, device, num_batches=5):
    original_mode = model.training; model.eval(); batch_global_entropies = []
    try:
        for i, (images, _) in enumerate(data_loader):
            if i >= num_batches: break
            images = images.to(device); _ = model(images)
            layer_entropies = []
            if model.latest_attn_maps:
                for attn_map in model.latest_attn_maps:
                    layer_entropies.append(robust_attention_entropy(attn_map.numpy()))
            if layer_entropies: batch_global_entropies.append(np.mean(layer_entropies))
    except Exception as e: print(f"Warning: Could not compute coarse global attn entropy. Error: {e}")
    finally: model.train(original_mode)
    return float(np.mean(batch_global_entropies)) if batch_global_entropies else 0.0


In [37]:

def get_dataloaders_and_config(dataset_name: str, batch_size: int, data_root: str):
    if dataset_name == 'mnist':
        config = {'img_size': 28, 'patch_size': 4, 'in_chans': 1, 'n_classes': 10}
        transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
        train_data = datasets.MNIST(root=data_root, train=True, download=True, transform=transform)
        test_data = datasets.MNIST(root=data_root, train=False, download=True, transform=transform)
    elif dataset_name == 'cifar10':
        config = {'img_size': 32, 'patch_size': 4, 'in_chans': 3, 'n_classes': 10}
        train_transform = transforms.Compose([transforms.RandomCrop(32,padding=4), transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010))])
        test_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010))])
        train_data = datasets.CIFAR10(root=data_root, train=True, download=True, transform=train_transform)
        test_data = datasets.CIFAR10(root=data_root, train=False, download=True, transform=test_transform)
    elif dataset_name == 'cifar100':
        config = {'img_size': 32, 'patch_size': 4, 'in_chans': 3, 'n_classes': 100}
        train_transform = transforms.Compose([transforms.RandomCrop(32,padding=4), transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize((0.5071,0.4867,0.4408),(0.2675,0.2565,0.2761))])
        test_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5071,0.4867,0.4408),(0.2675,0.2565,0.2761))])
        train_data = datasets.CIFAR100(root=data_root, train=True, download=True, transform=train_transform)
        test_data = datasets.CIFAR100(root=data_root, train=False, download=True, transform=test_transform)
    else: raise ValueError("Unknown or unsupported dataset")
    g = torch.Generator().manual_seed(42)
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=0, generator=g)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=0)
    return train_loader, test_loader, config


In [38]:

# ================== CELL 5: Training ==================
def run_experiment(model, optimizer, criterion, train_loader, test_loader, device, n_layers, epochs, log_csv_path, save_model_path, use_dynamic_rules: bool):
    model.to(device); logs_list = []; best_val_acc = -1.0; best_model_state = None
    history = {'val_loss': [], 'mid_entropy': []}
    patience_counter = 0; PATIENCE_LIMIT = 8; best_val_loss = float('inf'); stage_trigger_activated = False
    mid_layers_indices = list(range(n_layers // 2 - 1, n_layers // 2 + 2)) if n_layers > 3 else list(range(n_layers))
    EVAL_FREQ = 5

    for ep in (pbar_epoch := tqdm(range(epochs), desc="Epochs")):
        model.train()
        running_loss, train_correct, train_total, flat_grad = 0.0, 0, 0, None
        for bi, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device); optimizer.zero_grad()
            outputs = model(images); loss = criterion(outputs, labels); loss.backward(); optimizer.step()
            if (bi + 1) == len(train_loader): flat_grad = get_flat_grad(model)
            running_loss += loss.item(); train_correct += (outputs.argmax(dim=1) == labels).sum().item(); train_total += labels.size(0)

        train_loss = running_loss / len(train_loader); train_acc = train_correct / train_total
        val_loss, val_acc = evaluate_simple(model, test_loader, device)
        grad_entropy = robust_grad_entropy(flat_grad) if flat_grad is not None and flat_grad.numel() > 0 else 0.0
        coarse_attn_entropy = get_coarse_global_attn_entropy(model, test_loader, device, num_batches=5)

        log_row = {"epoch": ep + 1, "train_loss": train_loss, "val_loss": val_loss, "train_acc": train_acc, "val_acc": val_acc, "grad_entropy": grad_entropy, "coarse_global_attn_entropy": coarse_attn_entropy}

        if (ep + 1) % EVAL_FREQ == 0 or (ep + 1) == epochs:
            print(f"\n--- Running Full Evaluation for Epoch {ep+1} ---")
            full_metrics = evaluate_full_metrics(model, test_loader, device)
            history['mid_entropy'].append(np.mean([full_metrics.get('attn_entropy', [0.0]*n_layers)[i] for i in mid_layers_indices]))
            for key, value in full_metrics.items():
                if isinstance(value, list):
                    for i, v_item in enumerate(value): log_row[f"{key}_layer_{i}"] = v_item
                else: log_row[key] = value
            print(f"--- Full Evaluation for Epoch {ep+1} Complete ---")

        logs_list.append(log_row)
        if val_acc > best_val_acc:
            best_val_acc = val_acc; best_model_state = model.state_dict()

        if use_dynamic_rules:
            if len(history['mid_entropy']) > 0:
                mid_entropy_slope = recent_slope(history['mid_entropy'], window=5)
                if val_loss < best_val_loss - 1e-4:
                    best_val_loss = val_loss; patience_counter = 0
                else: patience_counter += 1
                if patience_counter >= PATIENCE_LIMIT and abs(mid_entropy_slope) < 1e-3:
                    print(f"\n[INFO] Early Stopping Triggered at Epoch {ep+1}.")
                    break
                if not stage_trigger_activated and len(history['mid_entropy']) * EVAL_FREQ > 12:
                    is_plateau = abs(recent_slope(history['mid_entropy'], window=2)) < 1e-3
                    was_declining = (history['mid_entropy'][0] - history['mid_entropy'][-1]) > 0.05
                    if is_plateau and was_declining:
                        stage_trigger_activated = True; print(f"\n[INFO] Stage Trigger at Epoch {ep+1}.")
                        for g in optimizer.param_groups: g['lr'] *= 0.2
                        layers_to_freeze = n_layers // 2
                        for i, layer in enumerate(model.encoder_layers):
                            if i < layers_to_freeze:
                                for param in layer.parameters(): param.requires_grad = False

        pbar_epoch.set_postfix_str(f"val_acc {val_acc:.3f} H_coarse {coarse_attn_entropy:.3f}")

    if best_model_state: torch.save(best_model_state, save_model_path)
    print("\n✅ Training complete. Best validation accuracy: {:.4f}".format(best_val_acc))
    logs_df = pd.DataFrame(logs_list); logs_df.to_csv(log_csv_path, index=False)
    print("📄 Full logs saved to:", log_csv_path)


In [ ]:

print("Copying dataset from GDrive to local Colab disk...")
gdrive_data_path = DATA_ROOT; local_data_path = "/tmp/vision_data"
os.makedirs(local_data_path, exist_ok=True)
!cp -r -n {gdrive_data_path}/* {local_data_path}/
print("Copy complete.")

# --- Setting ---
ALL_OPTIMIZER_CONFIGS = {"Adam_noWD": {"optimizer": torch.optim.Adam, "wd": 0.0}, "Adam_WD": {"optimizer": torch.optim.Adam, "wd": 1e-4}}
MODEL_CONFIGS = {"M_Base": {"n_layers": 8, "hidden_dim": 128},"M_Deep":   {"n_layers": 12, "hidden_dim": 128}}
SEED_LIST = [16, 42, 123, 1234]
DATASETS_TO_RUN = ["cifar10"]
EPOCHS_PER_DATASET = {"mnist": 30, "cifar10": 100, "cifar100": 120}
EXPERIMENT_MODES = ["control", "dynamic"]
criterion = nn.CrossEntropyLoss(); print("Criterion initialized.")

for dataset_to_run in DATASETS_TO_RUN:
    epochs_to_run = EPOCHS_PER_DATASET[dataset_to_run]
    for mode in EXPERIMENT_MODES:
        print("\n" + "#"*60 + f"\n# RUNNING DATASET '{dataset_to_run}' IN MODE: {mode.upper()}\n" + "#"*60)
        for optim_name, optim_config in ALL_OPTIMIZER_CONFIGS.items():
            for model_name, model_config in MODEL_CONFIGS.items():
                for seed in SEED_LIST:
                    experiment_name = f"D_{dataset_to_run}_M_{model_name}_O_{optim_name}_S_{seed}_T_{mode}"
                    log_path = Path(OUTPUT_ROOT) / "grid_search_stable" / f"{experiment_name}_logs.csv"
                    ckpt_path = Path(OUTPUT_ROOT) / "grid_search_stable" / f"{experiment_name}_best.pth"
                    if log_path.exists():
                        print(f"✅ SKIPPING: {log_path.stem} already exists.")
                        continue
                    print("\n" + "="*50 + f"\n▶️ STARTING: {experiment_name}\n" + "="*50)
                    log_path.parent.mkdir(parents=True, exist_ok=True)
                    set_seed(seed)
                    train_loader, test_loader, data_config = get_dataloaders_and_config(dataset_to_run, 128, local_data_path)
                    model = SimpleTransformerClassifier(img_size=data_config['img_size'], patch_size=data_config['patch_size'], in_chans=data_config['in_chans'], n_heads=8, n_layers=model_config['n_layers'], hidden_dim=model_config['hidden_dim'], n_classes=data_config['n_classes'])
                    optimizer = optim_config["optimizer"](model.parameters(), lr=cfg.LEARNING_RATE, weight_decay=optim_config["wd"])

                    use_dynamic_rules = (mode == "dynamic")
                    run_experiment(model, optimizer, criterion, train_loader, test_loader, device, model_config['n_layers'], epochs_to_run, log_path, ckpt_path, use_dynamic_rules)

                    del model, optimizer, train_loader, test_loader; gc.collect()
                    if torch.cuda.is_available(): torch.cuda.empty_cache()

print("\n🚀 All specified experiments are complete.")